In [0]:
%sql
create or replace table policyprojcatalog.gold.sales_by_policytype_month(
  policy_type string,
  sale_month string,
  total_premium integer,
  updated_timestamp timestamp
)using delta
location 'abfss://goldlayer@projpolicysytem.dfs.core.windows.net/sales_by_policytype_month'

In [0]:
%sql
create or replace table policyprojcatalog.gold.sales_by_policytype_status(
  policy_type string,
  claim_status string,
  total_claim integer,
  total_claim_amount integer,
  updated_timestamp timestamp
)using delta
location 'abfss://goldlayer@projpolicysytem.dfs.core.windows.net/sales_by_policytype_status'

In [0]:
%sql
create or replace table policyprojcatalog.gold.claim_analysis(
  policy_type string,
  claim_status string,
  avg_claim_amount integer,
  max_claim_amount integer,
   min_claim_amount integer,
   total_claims integer,
  updated_timestamp timestamp
)using delta
location 'abfss://goldlayer@projpolicysytem.dfs.core.windows.net/claim_analysis'

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_gold_sales_by_policytype_month AS
SELECT
  policy_type,
  DATE_FORMAT(start_date, 'yyyy-MM') AS sale_month,
  CAST(SUM(premium) AS INT) AS total_premium
FROM policyprojcatalog.silver.policy
WHERE policy_type IS NOT NULL
  AND start_date IS NOT NULL
  AND premium IS NOT NULL
GROUP BY
  policy_type,
  DATE_FORMAT(start_date, 'yyyy-MM');

In [0]:
%sql
MERGE INTO policyprojcatalog.gold.sales_by_policytype_month AS T
USING vw_gold_sales_by_policytype_month AS S
ON T.policy_type = S.policy_type
AND T.sale_month = S.sale_month

WHEN MATCHED THEN UPDATE SET
  T.total_premium = S.total_premium,
  T.updated_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  policy_type,
  sale_month,
  total_premium,
  updated_timestamp
)
VALUES (
  S.policy_type,
  S.sale_month,
  S.total_premium,
  current_timestamp()
);